# 12 — The hardware program, checked end to end before submission

The validated stub (noise injected in-program) becomes a device
program without changing its physics: the same memory experiment, built as QIR from the
code interface, with optional seeded injection. Before spending credits, the exact program
runs on local Selene and must pass the same backend contract as every simulator.

**Source:** `qec_timing.hardware` (`qir`, `selene_local`, `records`, `providers`, `cli`);
programs in `results/hardware/programs/`. This notebook reads their metadata; the local
check runs in the H2 env (`hardware.cli local`), recorded below.

In [1]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from qec_timing import plotting as P
P.style()
R = Path("..") / "results"

In [2]:
for p in sorted((R / "hardware" / "programs").glob("*.json")):
    m = json.loads(p.read_text())
    print(f"{p.stem:<16} {m['n_qubits']:>3} qubits  {m['qir_bitcode_bytes']/1e3:6.1f} kB  "
          f"{sum(m['qir_calls'].values()):>6} calls  p_inject {m['p_inject_effective']:.7f}  "
          f"basis {m['spec']['basis']}, rounds {m['spec']['schedule'].get('n_rounds')}")

d3_X_r2_p0.05     10 qubits     9.0 kB     302 calls  p_inject 0.0500002  basis X, rounds 2
d5_X_r5_p0        26 qubits    33.5 kB    1778 calls  p_inject 0.0000000  basis X, rounds 5


**The path, and what each step guarantees** (all *measured* in the test suite, `tests/test_hardware.py`):

| step | command (H2 env) | checks |
| --- | --- | --- |
| build | `hardware.cli build --d 3 --basis X --rounds 2 --p-inject 0.05` | QIR validated; byte-identical to the original Stage 7a program |
| local | `hardware.cli local <program>.bc` | backend contract on local Selene: no detections without noise, seeds reach the noise, round-0 detector rate inside the 99.9% binomial interval |
| price | `hardware.cli aqora-check` / `nexus-cost` | syntax checker only: validates and prices, executes nothing |
| run | `hardware.cli nexus-submit --confirm` | emulator first; refuses without `--confirm` |
| fetch, store | `hardware.cli nexus-fetch`, `store` | raw results kept verbatim; converted to a stored run, decoded like any simulation |

**Design choices that matter on a device.**
- Injection uses an integer Bernoulli (`random_int_bounded(2^20) < round(p·2^20)`), because
  hugr-qir 0.1.2 cannot lower `RandomFloat`; the effective p is recorded.
- When the program injects its own dephasing, the emulator's idle dephasing is switched off
  by default, so the two are not double-counted (`providers.nexus_submit`).
- Extraction order is hook-safe for the surface code (`codes/surface.py`): it only matters
  with gate noise, which is exactly the device case.